# Day 12 — Feature engineering & Feature Store

Snowflake Notebook on `LEARN_WH`. Features are computed from **prior** orders only,
then registered as `Entity` + `FeatureView` so `generate_training_set` can as-of join
labels without pulling future information into the past.


## 1. Explore the data

`GOLD.CUSTOMER_ORDER_EVENTS` is all TPC-H SF1 orders for customers 1–400
(3,926 rows, 267 customers, ~15 orders each). A random `SAMPLE` of orders would
leave ~1 order per customer and make prior-history features useless.


In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, count, when, lit, lag, datediff, ln, coalesce, sum as ssum, min as smin, avg
from snowflake.snowpark.window import Window
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode
from snowflake.ml.modeling.preprocessing import OrdinalEncoder
from snowflake.ml.modeling.ensemble import RandomForestClassifier
from snowflake.ml.modeling.metrics import accuracy_score

session = get_active_session()
src = session.table("RETAIL_LAKEHOUSE.GOLD.CUSTOMER_ORDER_EVENTS")
src.describe().show()
src.select([count(when(col(c).is_null(), 1)).alias(c) for c in src.columns]).show()
print("rows", src.count(), "customers", src.select("CUSTOMER_ID").distinct().count())


EDA (live run): no nulls. `ORDER_VALUE` is right-skewed (min 1,131, mean 151,821,
max 449,551) — log-scale it. `SHIP_PRIORITY` is constantly 0 (drop it). Label
`IS_FULFILLED` is ~50/50.


## 2. Engineer features (no current-order leakage)

- **Numeric:** `LOG_PRIOR_SPEND` = `LN(1 + sum of ORDER_VALUE before this order)`
- **Time-based:** `DAYS_SINCE_FIRST_ORDER`
- **Categorical:** `PRIORITY_ORDINAL` = OrdinalEncoder of the **previous** order's priority

Window frame is `ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING` so the labeled
order's own revenue/status never enters the feature row.


In [ ]:
w = Window.partition_by("CUSTOMER_ID").order_by("TS", "ORDER_ID")
feat = (
    src.with_column("PRIOR_SPEND", coalesce(ssum("ORDER_VALUE").over(w.rows_between(Window.unboundedPreceding, -1)), lit(0.0)))
       .with_column("FIRST_TS", smin("TS").over(Window.partition_by("CUSTOMER_ID")))
       .with_column("PRIOR_PRIORITY", lag("ORDER_PRIORITY", 1).over(w))
)
feat = feat.with_column("DAYS_SINCE_FIRST_ORDER", datediff("day", col("FIRST_TS"), col("TS")))
feat = feat.with_column("LOG_PRIOR_SPEND", ln(lit(1.0) + col("PRIOR_SPEND")))
feat = feat.with_column("PRIOR_PRIORITY_FILLED", coalesce(col("PRIOR_PRIORITY"), lit("NONE")))

ordenc = OrdinalEncoder(
    input_cols=["PRIOR_PRIORITY_FILLED"],
    output_cols=["PRIORITY_ORDINAL"],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)
encoded = ordenc.fit(feat).transform(feat)

feature_table = encoded.select("CUSTOMER_ID", "TS", "LOG_PRIOR_SPEND", "DAYS_SINCE_FIRST_ORDER", "PRIORITY_ORDINAL")
feature_table.write.mode("overwrite").save_as_table("RETAIL_LAKEHOUSE.GOLD.CUSTOMER_FEATURES")
encoded.select("CUSTOMER_ID", "TS", "IS_FULFILLED").write.mode("overwrite").save_as_table("RETAIL_LAKEHOUSE.GOLD.CUSTOMER_LABELS")
session.table("RETAIL_LAKEHOUSE.GOLD.CUSTOMER_FEATURES").limit(5).show()


## 3–4. Register Entity and FeatureView

`Entity` is the join key (`CUSTOMER_ID`). `FeatureView` is the versioned feature
table with `refresh_freq='1 day'` (backed by a Dynamic Table).


In [ ]:
fs = FeatureStore(
    session=session,
    database="RETAIL_LAKEHOUSE",
    name="FEATURE_STORE",
    default_warehouse="LEARN_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
customer_entity = Entity(name="CUSTOMER", join_keys=["CUSTOMER_ID"], desc="Retail customer join key")
fs.register_entity(customer_entity)
fs.list_entities().show()

fv = FeatureView(
    name="customer_features",
    entities=[customer_entity],
    feature_df=session.table("RETAIL_LAKEHOUSE.GOLD.CUSTOMER_FEATURES"),
    timestamp_col="TS",
    refresh_freq="1 day",
    desc="log prior spend, days since first order, ordinal last priority",
    warehouse="LEARN_WH",
)
registered_fv = fs.register_feature_view(fv, version="1", block=True, overwrite=True)
print(registered_fv.name, registered_fv.version, registered_fv.status)


## 5. Point-in-time training set

`generate_training_set(..., spine_timestamp_col='TS')` as-of joins each label row
to the latest feature snapshot with `feature.TS <= label.TS`.

A plain `JOIN ON CUSTOMER_ID` would attach the customer's *latest* features
(including spend that happened after the label) to historical labels — that is
label leakage. The Feature Store join is leakage-free by construction.


In [ ]:
spine_df = session.table("RETAIL_LAKEHOUSE.GOLD.CUSTOMER_LABELS")
training_set = fs.generate_training_set(
    spine_df=spine_df,
    features=[registered_fv],
    spine_timestamp_col="TS",
    spine_label_cols=["IS_FULFILLED"],
    save_as="PIT_TRAINING_SET",
)
training_set.limit(8).show()
print("rows", training_set.count(), "cols", training_set.columns)


## 6. Retrain vs Day 11 ad-hoc columns

Day 11 used same-order `ORDER_VALUE` / priority (accuracy **0.515** on this
cohort — no better than chance). Day 12 trains the same RandomForestClassifier
on PIT features (accuracy **0.963**).

Caveat: TPC-H `F` vs `O` is mostly recency, so `DAYS_SINCE_FIRST_ORDER` alone
scores 0.957. PIT is still the right *join*; the metric jump is not magic.


In [ ]:
pit = session.table("RETAIL_LAKEHOUSE.FEATURE_STORE.PIT_TRAINING_SET")
train_df, test_df = pit.random_split([0.8, 0.2], seed=42)
feature_cols = ["LOG_PRIOR_SPEND", "DAYS_SINCE_FIRST_ORDER", "PRIORITY_ORDINAL"]
model = RandomForestClassifier(
    input_cols=feature_cols,
    label_cols=["IS_FULFILLED"],
    output_cols=["PREDICTED"],
    n_estimators=20,
    max_depth=6,
    random_state=42,
)
model.fit(train_df)
pred = model.predict(test_df)
print("day12_pit_accuracy", accuracy_score(df=pred, y_true_col_names="IS_FULFILLED", y_pred_col_names="PREDICTED"))
